# ⚡ Simple NVIDIA Nemotron-3-Nano-30B Inference via vLLM

This notebook provides a clean, single-prompt inference script for the **NVIDIA-Nemotron-3-Nano-30B** base model. It is directly inspired by the environment setup, package extraction, and vLLM configuration found in the competition baseline submissions.

## 🛠️ Step 1: Environment Setup & Package Unpacking

We uninstall the default PyTorch stack and unpack the custom Blackwell-optimized environment packages provided via Kaggle's metric utility script.

In [2]:
import subprocess
import sys

commands = [
    "uv pip uninstall torch torchvision torchaudio",
    "tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
]

for cmd in commands:
    print(f"Running: {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# Prepend the unpacked environment directory to python search path
sys.path.insert(0, "/tmp")

Running: uv pip uninstall torch torchvision torchaudio


Using Python 3.12.13 environment at: /usr
Uninstalled 3 packages in 791ms
 - torch==2.10.0+cu128
 - torchaudio==2.10.0+cu128
 - torchvision==0.25.0+cu128


Running: tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell


## 📥 Step 2: Configure Environment Variables & Load Model

In [3]:
import os

# Set Triton and Transformers flags for offline execution
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

from vllm import LLM, SamplingParams
import kagglehub

# Download or locate the base model path
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print(f"Using model path: {MODEL_PATH}")

# Initialize vLLM Engine
llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=64,
    gpu_memory_utilization=0.85,
    dtype="auto",
    max_model_len=8192,
    trust_remote_code=True,
    enable_lora=False,  # base model only
)

2026-06-07 11:52:39.468819: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780833159.666992     163 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780833159.730415     163 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780833160.246936     163 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780833160.246950     163 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780833160.246951     163 computation_placer.cc:177] computation placer alr

Using model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
INFO 06-07 11:52:51 [utils.py:238] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 64, 'disable_log_stats': True, 'model': '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 06-07 11:53:18 [model.py:531] Resolved architecture: NemotronHForCausalLM
INFO 06-07 11:53:18 [model.py:1554] Using max model len 8192
INFO 06-07 11:53:18 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 06-07 11:53:18 [config.py:618] Updating mamba_ssm_cache_dtype to 'float32' for NemotronH model
INFO 06-07 11:53:19 [config.py:544] Setting attention block size to 2096 tokens to ensure that attention page size is >= mamba page size.
INFO 06-07 11:53:19 [config.py:575] Padding mamba page size by 0.58% to ensure that mamba page size and attention page size are exactly equal.
INFO 06-07 11:53:19 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 06-07 11:53:19 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


2026-06-07 11:53:25.588698: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780833205.599070     637 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780833205.602142     637 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780833205.609701     637 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780833205.609720     637 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780833205.609721     637 computation_placer.cc:177] computation placer alr

...

(EngineCore_DP0 pid=637) INFO 06-07 12:01:54 [default_loader.py:293] Loading weights took 503.63 seconds
(EngineCore_DP0 pid=637) INFO 06-07 12:01:55 [gpu_model_runner.py:4364] Model loading took 58.91 GiB memory and 504.063110 seconds
(EngineCore_DP0 pid=637) INFO 06-07 12:01:57 [backends.py:916] Using cache directory: /root/.cache/vllm/torch_compile_cache/1a46b4c999/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=637) INFO 06-07 12:01:57 [backends.py:976] Dynamo bytecode transform time: 2.13 s
(EngineCore_DP0 pid=637) INFO 06-07 12:01:59 [backends.py:350] Cache the graph of compile range (1, 16384) for later use


(EngineCore_DP0 pid=637) /tmp/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
(EngineCore_DP0 pid=637)   warnings.warn(


(EngineCore_DP0 pid=637) WARNING 06-07 12:02:01 [fused_moe.py:1093] Using default MoE config. Performance might be sub-optimal! Config file not found at /tmp/vllm/model_executor/layers/fused_moe/configs/E=128,N=1856,device_name=NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition.json
(EngineCore_DP0 pid=637) INFO 06-07 12:02:05 [backends.py:366] Compiling a graph for compile range (1, 16384) takes 7.80 s
(EngineCore_DP0 pid=637) INFO 06-07 12:02:05 [monitor.py:35] torch.compile takes 10.33 s in total
(EngineCore_DP0 pid=637) INFO 06-07 12:02:05 [decorators.py:580] saving AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/d9179ee62f2c548ab6be329dc16bdddb4ec7e42dfd5fd2e049e0aa52430b5293/rank_0_0/model
(EngineCore_DP0 pid=637) INFO 06-07 12:02:06 [decorators.py:588] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/d9179ee62f2c548ab6be329dc16bdddb4ec7e42dfd5fd2e049e0aa52430b5293/rank_0_0/model
(EngineCore_DP0 pid=637) INFO 06-07 

(EngineCore_DP0 pid=637) 2026-06-07 12:02:11,249 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=637) 2026-06-07 12:02:11,298 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 19/19 [00:02<00:00,  7.03it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 11/11 [00:40<00:00,  3.64s/it]


(EngineCore_DP0 pid=637) INFO 06-07 12:02:55 [gpu_model_runner.py:5386] Graph capturing finished in 44 secs, took -1.27 GiB
(EngineCore_DP0 pid=637) INFO 06-07 12:02:55 [core.py:282] init engine (profile, create kv cache, warmup model) took 59.96 seconds
(EngineCore_DP0 pid=637) INFO 06-07 12:02:56 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 06-07 12:02:56 [llm.py:388] Supported tasks: ['generate']


## 💬 Step 3: Define Inference Function & Run Prompt

In [4]:
def generate_response(prompt_text: str, temperature: float = 0.0, max_tokens: int = 7680) -> str:
    """Formats the prompt using the chat template and generates a response from the vLLM engine."""
    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=1.0,
        max_tokens=max_tokens,
    )
    tokenizer = llm.get_tokenizer()
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = llm.generate([prompt], sampling_params=sampling_params)
    return outputs[0].outputs[0].text

# Example Usage:
prompt = "Convert the number 42 into Roman numerals."
print(f"Prompt: {prompt}\n")

print("Generating response...")
response = generate_response(prompt)

print("\n" + "="*40 + " RESPONSE " + "="*40)
print(response)
print("="*90)

Prompt: Convert the number 42 into Roman numerals.

Generating response...


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


======================================== RESPONSE ========================================
The user asks: "Convert the number 42 into Roman numerals." Straightforward: 42 = XLII. Provide answer.
</think>
The Roman numeral for 42 is **XLII**.


# 📊 Step 4: Run Diagnostic Batch Inference on Dataset

We load the training dataset, prepare prompts using the custom system prompt and template, run fast batch generation using the vLLM engine, and output diagnostic statistics including accuracy and failure cases.

## 💬 Step 1: Define Prompts and Chat Formatting

In [5]:
SYSTEM_PROMPT = """You are an expert pattern recognition solver competing in a reasoning challenge.

You will be given a puzzle from Alice's Wonderland where a secret transformation rule converts inputs to outputs.

Your approach MUST follow these steps inside <think> tags:
1. HYPOTHESIZE: Study each input-output example and state the rule clearly
2. VERIFY: Apply your rule to every example to confirm it holds
3. APPLY: Use the confirmed rule on the test input

Your final answer MUST be inside \\boxed{} and nothing after it.

Format:
<think>
HYPOTHESIZE: ...
VERIFY: ...
APPLY: ...
</think>
\\boxed{your_answer}"""

def build_prompt(puzzle_text: str) -> list[dict]:
    """
    puzzle_text is the raw 'prompt' column from train.csv / test.csv
    Returns messages list ready for vLLM or Claude API
    """
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": puzzle_text}
    ]

## 📁 Step 4: Load Dataset & Configure Sampling

In [6]:
import pandas as pd
import os

# Set dataset path (change file name as needed)
data_path = "DATA/train.csv" if os.path.exists("DATA/train.csv") else "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"

# --- CONFIGURATION: Set number of samples to evaluate (e.g. 4, 30, or None for full dataset) ---
DIAGNOSTIC_SAMPLES = 4  

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path, dtype=str)

if DIAGNOSTIC_SAMPLES is not None:
    DIAGNOSTIC_SAMPLES = min(DIAGNOSTIC_SAMPLES, len(df))
    df_sample = df.head(DIAGNOSTIC_SAMPLES).copy()
    print(f"Sampled {DIAGNOSTIC_SAMPLES} rows from the dataset.")
else:
    DIAGNOSTIC_SAMPLES = len(df)
    df_sample = df.copy()
    print(f"Loaded the full dataset of {DIAGNOSTIC_SAMPLES} rows.")

Loading dataset from: /kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
Sampled 4 rows from the dataset.


## 📝 Step 5: Format Prompts & Inspect a Sample Prompt

In [7]:
print(f"Preparing prompts for {DIAGNOSTIC_SAMPLES} examples using SYSTEM_PROMPT...")
tokenizer = llm.get_tokenizer()
prompts = []
for idx, row in df_sample.iterrows():
    messages = build_prompt(row["prompt"])
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(input_text)

# Inspect the very first formatted prompt
print("\n" + "="*40 + " INSPECT FIRST FORMATTED PROMPT " + "="*40)
print(prompts[0])
print("="*112)

Preparing prompts for 4 examples using SYSTEM_PROMPT...

======================================== INSPECT FIRST FORMATTED PROMPT ========================================
<|im_start|>system
You are an expert pattern recognition solver competing in a reasoning challenge.

You will be given a puzzle from Alice's Wonderland where a secret transformation rule converts inputs to outputs.

Your approach MUST follow these steps inside <think> tags:
1. HYPOTHESIZE: Study each input-output example and state the rule clearly
2. VERIFY: Apply your rule to every example to confirm it holds
3. APPLY: Use the confirmed rule on the test input

Your final answer MUST be inside \boxed{} and nothing after it.

Format:
<think>
HYPOTHESIZE: ...
VERIFY: ...
APPLY: ...
</think>
\boxed{your_answer}<|im_end|>
<|im_start|>user
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly m

## 🚀 Step 6: Run Batch Inference & View Raw Response

In [8]:
# Sampling parameters matching evaluation environment
sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=7680
)

print(f"Running batch generation via vLLM on {len(prompts)} prompts...")
outputs = llm.generate(prompts, sampling_params)
print("Batch inference complete!")

# Print raw, unparsed response of the first generated sample
print("\n" + "="*40 + " RAW RESPONSE OF SAMPLE 1 " + "="*40)
print(outputs[0].outputs[0].text)
print("="*105)

Running batch generation via vLLM on 4 prompts...


Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Batch inference complete!

======================================== RAW RESPONSE OF SAMPLE 1 ========================================
We need to find transformation rule. 8-bit inputs and outputs. Let's list them:

1) 01010001 -> 11011101
2) 00001001 -> 01101101
3) 00010101 -> 01010101
4) 11111111 -> 10000001
5) 10011101 -> 01000101
6) 00111011 -> 00001001
7) 10111101 -> 00000101
8) 00100110 -> 10110011

Goal: output for 00110100.

We need to hypothesize rule. Let's examine patterns.

Write bits positions maybe index 7..0 (most significant to least). Let's label bits as b7 b6 b5 b4 b3 b2 b1 b0.

Input1: 0 1 0 1 0 0 0 1 -> output: 1 1 0 1 1 1 0 1

Input2: 0 0 0 0 1 0 0 1 -> output: 0 1 1 0 1 1 0 1

Input3: 0 0 0 1 0 1 0 1 -> output: 0 1 0 1 0 1 0 1

Input4: 1 1 1 1 1 1 1 1 -> output: 1 0 0 0 0 0 0 1

Input5: 1 0 0 1 1 1 0 1 -> output: 0 1 0 0 0 1 0 1

Input6: 0 0 1 1 1 0 1 1 -> output: 0 0 0 0 1 0 0 1

Input7: 1 0 1 1 1 1 0 1 -> output: 0 0 0 0 0 1 0 1

Input8: 0 0 1 0 0 1 1 0 -> output

## 📊 Step 7: Parse Results & Calculate Performance

In [9]:
import re

import re

def extract_boxed_content(text):
    """
    Extracts the content inside \\boxed{...} LaTeX blocks.
    Prioritizes the last matching boxed structure, handling basic nested brackets
    and unclosed boxed templates at the end.
    """
    if not text:
        return "NOT_FOUND"
    matches = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()
    return "NOT_FOUND"

def parse_model_output(output_text):
    """
    Parses the generated response to separate thinking process and the final answer.
    Handles prefilled <think> tags (which aren't in output_text) and unfinished responses.
    """
    thought = ""
    answer_raw = output_text
    
    if "</think>" in output_text:
        parts = output_text.split("</think>", 1)
        if "<think>" in parts[0]:
            sub_parts = parts[0].split("<think>", 1)
            thought = sub_parts[1].strip()
        else:
            thought = parts[0].strip()
        answer_raw = parts[1].strip()
    else:
        # If the model never closed the think tag, the entire generated text is reasoning
        thought = output_text.strip()
        answer_raw = ""
            
    # Extract boxed answer from the ENTIRE generated text first
    extracted_answer = extract_boxed_content(output_text)
    return thought, answer_raw, extracted_answer

print("Parsing generated results...")
results = []
has_answers = "answer" in df_sample.columns

for idx, output in enumerate(outputs):
    row = df_sample.iloc[idx]
    generated_text = output.outputs[0].text
    thought, raw_response, final_answer = parse_model_output(generated_text)
    
    is_correct = None
    ground_truth = None
    if has_answers:
        ground_truth = row["answer"]
        is_correct = (final_answer.strip().lower() == str(ground_truth).strip().lower())
    
    # Track sequence length and whether it finished or timed out
    num_generated_tokens = len(output.outputs[0].token_ids)
    finish_reason = getattr(output.outputs[0], "finish_reason", "unknown")
    
    results.append({
        "id": row["id"],
        "prompt": row["prompt"],
        "ground_truth": ground_truth,
        "model_thought": thought,
        "model_raw_response": raw_response,
        "model_answer": final_answer,
        "is_correct": is_correct,
        "tokens_generated": num_generated_tokens,
        "finish_reason": finish_reason
    })

df_results = pd.DataFrame(results)

# Print diagnostics about generation length
print("\nGeneration statistics:")
for idx, row in df_results.iterrows():
    has_think_end = "</think>" in outputs[idx].outputs[0].text
    print(f"Sample {idx} (ID: {row['id']}): Generated {row['tokens_generated']} tokens. Reason: '{row['finish_reason']}'. Closed </think>: {has_think_end}")

if has_answers:
    accuracy = df_results["is_correct"].mean()
    print(f"\nDiagnostic Accuracy on {DIAGNOSTIC_SAMPLES} samples: {accuracy * 100:.2f}%")
else:
    print(f"\nInference completed on {DIAGNOSTIC_SAMPLES} samples (No ground truth 'answer' column found).")

output_csv = "diagnostic_results_simple.csv"
df_results.to_csv(output_csv, index=False)
print(f"Diagnostic results saved to: {output_csv}")


Parsing generated results...

Diagnostic Accuracy on 4 samples: 25.00%
Diagnostic results saved to: diagnostic_results_simple.csv


## 📊 Step 8: View Results in Tabular Format

In [10]:
# Load generated CSV results in table format
df_view = pd.read_csv("diagnostic_results_simple.csv", dtype=str)

# Set display settings for clean table viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print(f"Loaded diagnostic results: {len(df_view)} rows.")
df_view.head(10)

Loaded diagnostic results: 4 rows.


,id,prompt,ground_truth,model_thought,model_raw_response,model_answer,is_correct
0,00066667,"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The trans...",10010111,NaN,We need to find transformation rule. 8-bit inputs and outputs. Let's list them:\n\n1) 01010001 -...,3,False
1,000b53cf,"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The trans...",01000011,NaN,We need to find transformation rule. Given 8-bit inputs and outputs. Let's list them:\n\n1) 1000...,0,False
2,00189f6a,"In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:\nucoov ...",cat imagines book,NaN,We need to find transformation rule mapping ciphertext to plaintext. Given examples: ciphertext ...,3,False
3,001b24c4,"In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some exam...",XXXVIII,NaN,We need to interpret the puzzle: numbers are converted into a different numeral system. Examples...,XXXVIII,True


## 🔍 Step 9: View a Specific Sample in Detail

In [11]:
def inspect_sample(index: int):
    if index < 0 or index >= len(df_results):
        print(f"Index out of range. Use 0 to {len(df_results) - 1}.")
        return
    
    row = df_results.iloc[index]
    print("=" * 100)
    print(f"🔍 INSPECTING SAMPLE AT INDEX {index} (ID: {row['id']})")
    print("=" * 100)
    print(f"\n[PROMPT]:\n{row['prompt']}")
    print("-" * 100)
    print(f"\n[MODEL THOUGHTS]:\n{row['model_thought']}")
    print("-" * 100)
    print(f"\n[MODEL EXTRACTED ANSWER]: {row['model_answer']}")
    print(f"[GROUND TRUTH]:           {row['ground_truth']}")
    print(f"[IS CORRECT]:             {row['is_correct']}")
    print(f"[TOKENS GENERATED]:       {row['tokens_generated']}")
    print(f"[FINISH REASON]:          {row['finish_reason']}")
    print("=" * 100)

# Change this index to view different samples!
inspect_sample(0)

🔍 INSPECTING SAMPLE AT INDEX 0 (ID: 00066667)

[PROMPT]:
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100
----------------------------------------------------------------------------------------------------

[MODEL THOUGHTS]:

----------------------------------------------------------------------------------------------------

[MODEL EXTRACTED ANSWER]: 3
[GROUND TRUTH]:           10010111
[IS CORRECT]:             False
